In [ ]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cpu",
)

messages = [
    {"role": "user", "content": "Hello! Can you introduce yourself?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)
outputs = model.generate(  # type: ignore
    **inputs,
    max_new_tokens=256,
    do_sample=False,  # temperature = 0.0
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(response)

Loading weights:   0%|          | 1/883 [00:01<18:59,  1.29s/it]

: 

In [1]:
import json


class GemmaTokenizer:
    def __init__(self, vocab: dict[str, int], merges: list[list[str]]):
        self.vocab = vocab
        self.idx_to_str = {i: s for s, i in vocab.items()}

        self.ranks = {}
        self.merges = {}

        for i, (s1, s2) in enumerate(merges):
            pair = (vocab[s1], vocab[s2])
            self.ranks[pair] = i
            self.merges[pair] = vocab[s1 + s2]

    def encode(self, text: str) -> list[int]:
        text = text.replace(" ", "▁")

        tokens = [self.vocab["<bos>"]]
        for char in text:
            if char in self.vocab:
                tokens.append(self.vocab[char])
            else:
                for byte in char.encode("utf-8", errors="replace"):
                    tokens.append(self.vocab[f"<0x{hex(byte)[2:].upper()}>"])

        while True:
            best_rank = float("inf")
            best_pair = None

            for pair in zip(tokens[1:], tokens[2:]):
                if ((rank := self.ranks.get(pair)) is not None) and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            i = 1
            while i < len(tokens) - 1:
                pair = (tokens[i], tokens[i + 1])
                if pair == best_pair:
                    tokens[i:i + 2] = [self.merges[pair]]
                i += 1

        return tokens

    def decode(self, tokens: list[int]) -> str:
        return (
            "".join(self.idx_to_str[tok] for tok in tokens)
        ).replace("▁", " ")


with open("./Models/gemma-3-1b-it/tokenizer.json", encoding="utf-8") as f:
    data = json.load(f)

tokenizer = GemmaTokenizer(data["model"]["vocab"], data["model"]["merges"])

In [19]:
text = "致命 BUG 解析\nc\n运行\nint cmp_merge(const void *a, const void *b) {\n    char *astr1 = ((struct Merge *)a)->str1;\n    char *astr2 = ((struct Merge *)b)->str2;\n    char *bstr1 = ((struct Merge *)a)->str1;\n    char *bstr2 = ((struct Merge *)b)->str2;\n    return strcmp(strcat(astr1, astr2), strcat(bstr1, bstr2));\n}\n这里不止一个错误，strcat直接修改原始字符串缓冲区，直接破坏 tokenizer 内部的 merge 字符串，造成堆内存越界、内存损坏、崩溃。\n1. 最核心灾难：strcat会修改传入的源字符串\nstrcat(dst, src)：把 src 追加到 dst 末尾，改写 dst 内存，不是返回新拼接字符串！\nc\n运行\nstrcat(astr1, astr2);\nastr1是你struct Merge里保存的原始字符串指针（BPE merge 的子串）。\n原本字符串：astr1 = \"ab\"，astr2=\"cd\"\nstrcat 之后：astr1指向的内存被改写变成 \"abcd\"！\n你直接把 tokenizer 存的原始 merge 字符串原地篡改了！\n堆内存被破坏；\n别的地方还在使用这些str1/str2指针，字符串内容已经被改得面目全非；\n如果字符串内存是只读（比如字符串字面量），直接段错误 crash。\n⚠️ strcat 不会分配新内存，它在原有缓冲区上追加。你没有临时缓冲区，直接修改原始数据。\n2. 变量复制写错（逻辑 bug）\nc\n运行\nchar *astr1 = ((struct Merge *)a)->str1;\nchar *astr2 = ((struct Merge *)b)->str2;   // ❌ 拿的是b的str2\nchar *bstr1 = ((struct Merge *)a)->str1;   // ❌ bstr1取成a的str1\nchar *bstr2 = ((struct Merge *)b)->str2;\n你本意：\na 项：a->str1 + a->str2\nb 项：b->str1 + b->str2\n现在代码完全取错字段，比较逻辑本身就是错的。\n3. qsort 比较函数的硬性规则\nqsort 的 cmp 回调绝对不能修改 a、b 指向的任何数据！\na、b只是只读的待比较元素，在回调内部写它们的内存属于未定义行为，直接内存崩坏。\n4. 拼接字符串的正确思路\n如果你想比较 a.str1+a.str2 和 b.str1+b.str2，不能修改原始字符串，有两条路：\n方案 1：不拼接，直接两次 strcmp（推荐，不需要堆分配，最快）\n直接先比 str1，str1 相等再比 str2，不需要拼接字符串。\nc\n运行\nint cmp_merge(const void *a_, const void *b_)\n{\n    const struct Merge *a = (const struct Merge *)a_;\n    const struct Merge *b = (const struct Merge *)b_;\n\n    int ret = strcmp(a->str1, b->str1);\n    if (ret != 0)\n        return ret;\n    return strcmp(a->str2, b->str2);\n}\n✅ 只读，不修改任何内存；适合 qsort /bsearch。\n这就是 BPE merge 按(str1,str2)二元组排序的标准写法。\n方案 2：非要拼接完整合并字符串做比较（不推荐，开销大）\n必须开辟临时缓冲区，不能碰原始字符串。\nqsort 回调里面尽量不要 malloc，频繁排序会疯狂分配释放，性能差还容易泄露。\nc\n运行\n// 不建议在cmp内部malloc！仅演示原理\nint cmp_merge(const void *a_, const void *b_)\n{\n    const struct Merge *a = (const struct Merge *)a_;\n    const struct Merge *b = (const struct Merge *)b_;\n\n    char bufA[256], bufB[256]; // 栈临时缓冲区，注意足够大小\n    snprintf(bufA, sizeof(bufA), \"%s%s\", a->str1, a->str2);\n    snprintf(bufB, sizeof(bufB), \"%s%s\", b->str1, b->str2);\n    return strcmp(bufA, bufB);\n}\n⚠️ 栈数组要有足够长度，如果 merge 字符串很长会栈溢出。优先用方案 1，不要拼接。\n5. 额外提醒 const\nqsort/bsearch 传给 cmp 的指针，逻辑上是只读，最好转成const struct Merge*，防止你不小心写内存，编译器也会帮你拦截错误修改。\n总结你崩溃链条\nqsort 调用 cmp_merge；\nstrcat(astr1, astr2) → 原地改写 tokenizer 原始 merge 字符串内存；\n堆元数据被破坏；\n后续任意内存操作：free、strcmp、再 qsort、bsearch → 直接段错误 / 内存报错。\n记住铁律：strcat会修改第一个参数的内存，如果你只是想要拼接后的字符串用于比较，千万不要把原始数据指针丢给 strcat。\n如果你需要，我可以把配套的bsearch查找函数完整一并给你，直接适配 Gemma BPE。"

tokenizer.encode(text) == [2, 219089, 207025, 178667, 107, 236755, 107, 27602, 107, 720, 90654, 236779, 19215, 236769, 1987, 2325, 808, 236746, 236764, 1142, 2325, 808, 236763, 236768, 642, 107, 140, 4873, 808, 77807, 236770, 578, 5960, 6076, 84039, 14178, 236746, 20662, 1714, 236770, 236793, 107, 140, 4873, 808, 77807, 236778, 578, 5960, 6076, 84039, 14178, 236763, 20662, 1714, 236778, 236793, 107, 140, 4873, 808, 236763, 1714, 236770, 578, 5960, 6076, 84039, 14178, 236746, 20662, 1714, 236770, 236793, 107, 140, 4873, 808, 236763, 1714, 236778, 578, 5960, 6076, 84039, 14178, 236763, 20662, 1714, 236778, 236793, 107, 140, 2060, 122348, 236769, 1714, 9307, 236769, 77807, 236770, 236764, 20588, 236778, 779, 165764, 236769, 236763, 1714, 236770, 236764, 518, 1714, 236778, 2697, 107, 236783, 107, 21102, 178990, 5095, 36525, 236900, 1714, 9307, 17583, 33952, 43765, 50912, 136191, 236900, 17583, 114448, 92875, 236743, 42724, 236918, 22223, 236743, 50912, 236900, 38359, 240600, 43557, 238356, 237929, 236951, 43557, 187316, 236951, 201392, 236924, 107, 236770, 236761, 73847, 43173, 241640, 238723, 237184, 1714, 9307, 237279, 33952, 148965, 236918, 238076, 50912, 107, 1714, 9307, 236769, 30711, 236764, 4474, 127936, 238061, 4474, 233781, 237238, 56137, 225717, 239662, 236900, 238035, 238214, 56137, 236743, 43557, 236900, 11533, 29153, 237365, 192917, 50912, 237354, 107, 236755, 107, 27602, 107, 1714, 9307, 236769, 77807, 236770, 236764, 20588, 236778, 626, 107, 77807, 236770, 169757, 6076, 84039, 237652, 30826, 236918, 43765, 50912, 114055, 237221, 236799, 4110, 22223, 10363, 237313, 239586, 21719, 107, 73637, 50912, 237184, 77807, 236770, 578, 623, 596, 140493, 77807, 236778, 718, 2692, 236775, 107, 1714, 9307, 236743, 21569, 237184, 77807, 236770, 108983, 236918, 43557, 237759, 238035, 238214, 78216, 623, 200500, 236775, 237354, 107, 237408, 17583, 238061, 92875, 236743, 237797, 236918, 43765, 22223, 236743, 50912, 230516, 246725, 238035, 237089, 237354, 107, 240600, 43557, 237759, 114448, 237996, 107, 238147, 46611, 81543, 5938, 16956, 1714, 236770, 236786, 1714, 236778, 114055, 236900, 50912, 13687, 176300, 238035, 237439, 237360, 237472, 237450, 237940, 237996, 107, 9522, 50912, 43557, 237026, 237787, 238813, 237221, 30546, 50912, 237871, 237360, 237509, 19612, 17583, 238186, 36525, 19592, 236924, 107, 195096, 165764, 20883, 237279, 64513, 237365, 43557, 236900, 238010, 237075, 213454, 136191, 237152, 26835, 236924, 237408, 8939, 104099, 136191, 236900, 17583, 33952, 43765, 9676, 236924, 107, 236778, 236761, 236743, 32508, 68170, 238214, 238836, 237221, 68929, 13582, 237214, 107, 236755, 107, 27602, 107, 4873, 808, 77807, 236770, 578, 5960, 6076, 84039, 14178, 236746, 20662, 1714, 236770, 236793, 107, 4873, 808, 77807, 236778, 578, 5960, 6076, 84039, 14178, 236763, 20662, 1714, 236778, 236793, 139, 715, 236743, 242523, 236743, 239074, 15093, 236763, 236918, 1714, 236778, 107, 4873, 808, 236763, 1714, 236770, 578, 5960, 6076, 84039, 14178, 236746, 20662, 1714, 236770, 236793, 139, 715, 236743, 242523, 518, 1714, 236770, 237653, 237283, 236746, 236918, 1714, 236770, 107, 4873, 808, 236763, 1714, 236778, 578, 5960, 6076, 84039, 14178, 236763, 20662, 1714, 236778, 236793, 107, 237408, 237261, 237588, 237184, 107, 236746, 201279, 237184, 236746, 1160, 1714, 236770, 900, 496, 1160, 1714, 236778, 107, 236763, 201279, 237184, 236763, 1160, 1714, 236770, 900, 518, 1160, 1714, 236778, 107, 16157, 22148, 23595, 237653, 238836, 92527, 236900, 23533, 68929, 53449, 7888, 238836, 236918, 236924, 107, 236800, 236761, 3752, 10479, 236743, 23533, 151428, 239509, 237409, 41573, 107, 236809, 10479, 10363, 90654, 41540, 238250, 58577, 17055, 33952, 496, 236951, 236763, 64352, 237736, 236918, 24794, 9676, 237354, 107, 236746, 236951, 236763, 24735, 237787, 238813, 236918, 238628, 23533, 36152, 236900, 237075, 196588, 42724, 238214, 188947, 43557, 46859, 238195, 27428, 42701, 236900, 17583, 43557, 241126, 240369, 236924, 107, 236812, 236761, 236743, 192917, 50912, 236918, 49324, 104118, 107, 54112, 237690, 23533, 496, 236761, 1714, 236770, 236862, 236746, 236761, 1714, 236778, 26595, 518, 236761, 1714, 236770, 236862, 236763, 236761, 1714, 236778, 236900, 17055, 33952, 43765, 50912, 236900, 237078, 229453, 237844, 237184, 107, 34002, 236743, 236770, 237184, 237070, 192917, 236900, 17583, 135042, 122348, 237221, 42929, 236900, 72889, 240600, 64513, 236900, 237383, 238198, 237214, 107, 17583, 237804, 237650, 1540, 236770, 236900, 1714, 236770, 116814, 237520, 237921, 237650, 1540, 236778, 236900, 72889, 192917, 50912, 236924, 107, 236755, 107, 27602, 107, 720, 90654, 236779, 19215, 236769, 1987, 2325, 808, 236746, 25317, 1142, 2325, 808, 236763, 37090, 107, 236782, 107, 140, 1987, 2456, 84039, 808, 236746, 578, 568, 1987, 2456, 84039, 14178, 236746, 43497, 107, 140, 1987, 2456, 84039, 808, 236763, 578, 568, 1987, 2456, 84039, 14178, 236763, 43497, 108, 140, 720, 2461, 578, 122348, 236769, 236746, 1160, 1714, 236770, 236764, 518, 1160, 1714, 236770, 626, 107, 140, 584, 568, 4243, 2843, 236743, 236771, 236768, 107, 144, 2060, 2461, 236793, 107, 140, 2060, 122348, 236769, 236746, 1160, 1714, 236778, 236764, 518, 1160, 1714, 236778, 626, 107, 236783, 107, 240732, 110778, 238813, 236900, 237070, 33952, 24794, 43557, 237996, 52125, 3752, 10479, 965, 236763, 2305, 236924, 107, 92953, 603, 4110, 22223, 148898, 236769, 1714, 236770, 236764, 1714, 236778, 236768, 237725, 237798, 238193, 87250, 216106, 236076, 236924, 107, 34002, 236743, 236778, 237184, 237940, 237208, 192917, 54766, 97838, 50912, 237893, 23533, 237221, 237070, 42929, 236900, 237575, 239094, 237110, 237214, 107, 28031, 237575, 243544, 104099, 136191, 236900, 17055, 240991, 43765, 50912, 236924, 107, 236809, 10479, 41540, 238250, 46682, 108366, 25165, 44372, 236900, 187198, 87250, 237279, 139300, 64513, 78110, 236900, 33805, 238575, 237670, 30094, 241796, 239322, 236924, 107, 236755, 107, 27602, 107, 715, 20883, 40696, 237075, 24671, 42724, 58545, 237354, 238938, 133687, 70945, 107, 720, 90654, 236779, 19215, 236769, 1987, 2325, 808, 236746, 25317, 1142, 2325, 808, 236763, 37090, 107, 236782, 107, 140, 1987, 2456, 84039, 808, 236746, 578, 568, 1987, 2456, 84039, 14178, 236746, 43497, 107, 140, 1987, 2456, 84039, 808, 236763, 578, 568, 1987, 2456, 84039, 14178, 236763, 43497, 108, 140, 4873, 25076, 236776, 236840, 236778, 236810, 236825, 1604, 25076, 236799, 236840, 236778, 236810, 236825, 2312, 973, 236743, 242048, 104099, 136191, 236900, 18577, 81244, 44893, 107, 140, 12238, 8641, 236769, 13894, 236776, 236764, 15260, 236769, 13894, 236776, 779, 22389, 236751, 236908, 236751, 827, 496, 1160, 1714, 236770, 236764, 496, 1160, 1714, 236778, 626, 107, 140, 12238, 8641, 236769, 13894, 236799, 236764, 15260, 236769, 13894, 236799, 779, 22389, 236751, 236908, 236751, 827, 518, 1160, 1714, 236770, 236764, 518, 1160, 1714, 236778, 626, 107, 140, 2060, 122348, 236769, 13894, 236776, 236764, 25076, 236799, 626, 107, 236783, 107, 195096, 236743, 242048, 58905, 158321, 81244, 59738, 236900, 9522, 22223, 236743, 50912, 237614, 237839, 237279, 242048, 241964, 237191, 236924, 90039, 237105, 34002, 236743, 236770, 236900, 25165, 192917, 236924, 107, 236810, 236761, 236743, 173884, 79057, 1142, 107, 236809, 10479, 236786, 236763, 2305, 236743, 238237, 238113, 90654, 10363, 114055, 236900, 68929, 237152, 237026, 237787, 238813, 236900, 62566, 238307, 237283, 1987, 2456, 84039, 236829, 236900, 42304, 141412, 89130, 238214, 43557, 236900, 163344, 54218, 211956, 210945, 36525, 33952, 236924, 107, 68301, 237408, 201392, 239296, 238226, 107, 236809, 10479, 226516, 90654, 236779, 19215, 237996, 107, 1714, 9307, 236769, 77807, 236770, 236764, 20588, 236778, 236768, 16130, 45306, 237307, 238035, 238214, 92875, 236743, 43765, 22223, 236743, 50912, 43557, 237996, 107, 240600, 237798, 9676, 237759, 114448, 237996, 107, 116839, 71535, 43557, 17553, 237184, 6796, 236951, 42857, 236951, 237921, 3752, 10479, 236951, 236763, 2305, 16130, 173436, 238186, 36525, 965, 236743, 43557, 161723, 236924, 107, 160717, 239750, 239204, 237184, 1714, 9307, 237279, 33952, 62779, 21036, 236918, 43557, 236900, 54112, 24735, 34860, 192917, 42590, 50912, 28861, 23533, 236900, 114512, 25165, 238061, 43765, 9676, 114055, 241392, 238113, 165764, 236924, 107, 54112, 10042, 236900, 183868, 238061, 131659, 236918, 236763, 2305, 76114, 20149, 54766, 237009, 237853, 82611, 236900, 17583, 160266, 147224, 603, 4110, 236924]

True

In [1]:
from export import load_bin

model, tokenizer = load_bin("gemma-3-1b-pt.bin")

d:\terry\Documents\Repositories\Gemma.fsh\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import torch
import torch.nn.functional as F

FP16_MAX = torch.finfo(torch.float16).max

from model import precompute_freqs_cis, apply_rotary_emb, GemmaDecoderBlock

config = model.config

freqs_cis = precompute_freqs_cis(config.head_dim, 1024, config.local_theta)

def check(name, t, i):
    if torch.isinf(t).any() or torch.isnan(t).any():
        print(f"[layer {i}] {name}: inf={torch.isinf(t).any().item()} nan={torch.isnan(t).any().item()}, max_abs={t.abs().max().item()}")


def forward(token, pos, kv_cache):
    x = torch.tensor(token)

    x = model.embedding.call(x)
    x = x * config.embed_dim**0.5

    for i, layer in enumerate(model.layers):
        assert isinstance(layer, GemmaDecoderBlock)

        resid = x
        x = layer.norm1.call(x)
        check("norm1(x)", x, i)

        attn = layer.attn
        xq = attn.q_proj.call(x).view(config.n_heads, config.head_dim)
        xk = attn.k_proj.call(x).view(config.n_kv_heads, config.head_dim)
        xv = attn.v_proj.call(x).view(config.n_kv_heads, config.head_dim)
        check("q_proj(x)", xq, i)
        check("k_proj(x)", xk, i)
        check("v_proj(x)", xv, i)

        xq = attn.q_norm.call(xq)
        xk = attn.k_norm.call(xk)
        check("q_norm", xq, i)
        check("k_norm", xk, i)

        xq = apply_rotary_emb(xq.view(1, xq.size(0), 1, xq.size(1)), freqs_cis[pos]).squeeze(0, 2)
        xk = apply_rotary_emb(xk.view(1, xk.size(0), 1, xk.size(1)), freqs_cis[pos]).squeeze(0, 2)
        check("rope(xq)", xq, i)
        check("rope(xk)", xk, i)

        kv_cache[i][pos, 0] = xk
        kv_cache[i][pos, 1] = xv

        key = kv_cache[i][:pos + 1, 0]
        val = kv_cache[i][:pos + 1, 1]

        key = key.repeat_interleave(config.n_heads // config.n_kv_heads, 1)
        val = val.repeat_interleave(config.n_heads // config.n_kv_heads, 1)

        att = ((xq.unsqueeze(-2) @ key.unsqueeze(-2).mT) * attn.scaling).squeeze(-1, -2)
        check("att(xq, key)", att, i)
        att = att[:pos + 1].T
        att = att.softmax(-1)  # (n_heads, seq_len)
        out = (att.unsqueeze(1) @ val.transpose(0, 1)).squeeze(1)
        check("val(att)", out, i)
        x = attn.o_proj.call(out.view(-1))
        check("o_proj(out)", x, i)

        x = layer.norm2.call(x)
        check("norm2(x)", x, i)
        x += resid
        x = x.clamp(-FP16_MAX, FP16_MAX)
        check("resid(x) #1", x, i)
        resid = x
        x = layer.norm3.call(x)
        check("norm3(x)", x, i)

        ffwd = layer.ffwd
        gate = ffwd.gate_proj.call(x)
        check("gate_proj(x)", gate, i)
        gate = F.gelu(gate, approximate="tanh")
        check("gelu(gate)", gate, i)
        up = ffwd.up_proj.call(x)
        check("up_proj(x)", up, i)
        fuse = gate * up
        check("gate * up", fuse, i)
        x = ffwd.down_proj.call(gate * up)
        check("down_proj(fuse)", x, i)
        x = layer.norm4.call(x)
        check("norm4(x)", x, i)
        x += resid
        x = x.clamp(-FP16_MAX, FP16_MAX)
        check("resid(x) #2", x, i)

        print(f"Layer #{i}:")
        print(f"  {str(x)}")

    x = model.final_norm.call(x)
    return x @ model.embedding.weight.T


kv_cache = [
    torch.zeros(200, 2, config.n_kv_heads, config.head_dim).to(torch.float16)
    for _ in range(config.n_layers)
]

def generate():
    prompt = "Once upon a time"
    print(prompt, end="")

    i = 0
    tok = 0
    for tok in tokenizer.encode(prompt):
        tok = forward(tok, i, kv_cache).argmax().item()
        i += 1

    for i in range(i, 200):
        tok = forward(tok, i, kv_cache).argmax().item()
        print(tokenizer.decode([int(tok)]), end="")


forward(2, 0, kv_cache)


Layer #0:
  tensor([  1.6396,  -0.1893,   0.1589,  ...,   0.0555, -15.6562,   0.1144],
       dtype=torch.float16, grad_fn=<ClampBackward1>)
Layer #1:
  tensor([-73.6875,   0.7876,   0.7832,  ...,  -0.3738, -26.2812,   0.7983],
       dtype=torch.float16, grad_fn=<ClampBackward1>)
Layer #2:
  tensor([-7.3875e+01,  1.8477e+00,  8.4766e-01,  ..., -4.4434e-02,
        -2.3188e+01,  3.4883e+00], dtype=torch.float16,
       grad_fn=<ClampBackward1>)
Layer #3:
  tensor([-72.8750,   2.0059,   0.5225,  ...,  -0.2390, -18.3125,   5.7031],
       dtype=torch.float16, grad_fn=<ClampBackward1>)
Layer #4:
  tensor([-5.9656e+01,  3.3691e-02, -5.5566e-01,  ..., -1.7783e+00,
        -1.4562e+01,  5.9062e+00], dtype=torch.float16,
       grad_fn=<ClampBackward1>)
Layer #5:
  tensor([-17.6406,   1.3320,  -0.8813,  ...,  -7.6992, -10.8125,  12.9844],
       dtype=torch.float16, grad_fn=<ClampBackward1>)
Layer #6:
  tensor([-17.2188,  -0.1582,  -0.1498,  ...,  -5.5391, -10.6016,   9.2500],
       dtype=to

tensor([-40.1875,   6.2930,  -4.3867,  ..., -40.1250, -40.5000, -40.0938],
       dtype=torch.float16, grad_fn=<SqueezeBackward4>)